# LeetCode #116 — Populating Next Right Pointers in Each Node: nối từng tầng, bộ nhớ O(1)

**Mức độ:** Medium · **Chủ đề:** Linked List, Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val: int = 0, left: 'Node' = None, right: 'Node' = None, next: 'Node' = None):
        self.val = val
        self.left = left
        self.right = right
        self.next = next
"""
class Solution:
    def connect(self, root: 'Optional[Node]') -> 'Optional[Node]':
```

Mục tiêu của notebook:

1. Cách 1: BFS theo tầng (#102), nối các nút liền kề trong tầng — `O(w)` bộ nhớ.
2. Cách 2 (follow-up): dùng **chính các con trỏ `next` của tầng trên** để đi ngang, nối tầng dưới — `O(1)` bộ nhớ.
3. Hai loại nối trong **cây hoàn hảo**: `left.next = right` (cùng cha) và `right.next = next.left` (khác cha).
4. Phiên bản đệ quy.

# 1. Bài toán là gì?

Cho một **cây nhị phân hoàn hảo** (perfect binary tree): mọi lá cùng tầng, mọi nút trong có **đúng hai** con. Mỗi nút có thêm con trỏ `next`.

Gán `next` của mỗi nút trỏ tới nút **bên phải liền kề cùng tầng**. Nếu không có, `next = None`. Ban đầu mọi `next` là `None`.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 2^12 − 1]`.
- `-1000 <= Node.val <= 1000`

**Follow up (theo đề):** chỉ được dùng bộ nhớ phụ **hằng số**; ngăn xếp đệ quy được coi là không tính.

---

## Ví dụ

```text
        1                 1 -> None
       / \               / \
      2   3      ->     2 -> 3 -> None
     / \ / \           / \  / \
    4  5 6  7         4->5->6->7 -> None

LeetCode in theo next, '#' đánh dấu hết tầng:  [1, #, 2, 3, #, 4, 5, 6, 7, #]

root = []  ->  []
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Duyệt theo chiều rộng (BFS — Breadth-First Search)
3. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [6]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [7]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [8]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.3 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [9]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [10]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [11]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [12]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [13]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.4 Áp dụng vào bài này

Nối các nút **cùng tầng** từ trái sang phải — cách tự nhiên là **BFS theo tầng**. Để đạt `O(1)` bộ nhớ, ta coi mỗi tầng đã nối là một **danh sách liên kết** (qua con trỏ `next`) và duyệt nó để nối tầng bên dưới.

# 3. Nền tảng

In [14]:
from typing import List, Optional
from collections import deque
import random


class Node:
    def __init__(self, val=0, left=None, right=None, next=None):
        self.val = val
        self.left = left
        self.right = right
        self.next = next


def build_perfect(values: list) -> Optional[Node]:
    """values có độ dài 2^k - 1, theo level-order."""
    if not values:
        return None
    nodes = [Node(v) for v in values]
    for i in range(len(nodes)):
        if 2 * i + 2 < len(nodes):
            nodes[i].left, nodes[i].right = nodes[2 * i + 1], nodes[2 * i + 2]
    return nodes[0]


def serialize_next(root: Optional[Node]) -> list:
    """Đi theo next từ nút trái nhất mỗi tầng, như LeetCode in."""
    out, leftmost = [], root
    while leftmost:
        node = leftmost
        while node:
            out.append(node.val)
            node = node.next
        out.append("#")
        leftmost = leftmost.left
    return out

print(serialize_next(build_perfect([1, 2, 3, 4, 5, 6, 7])), "<- chưa nối, mỗi tầng chỉ thấy nút trái nhất")

[1, '#', 2, '#', 4, '#'] <- chưa nối, mỗi tầng chỉ thấy nút trái nhất


## Đánh số nút trong cây hoàn hảo

Nếu đánh số theo tầng từ 0 như heap: con của `i` là `2i + 1` và `2i + 2`. Cây hoàn hảo `k` tầng có `2^k − 1` nút.

# 4. Cách 1: BFS theo tầng

Dùng khung #102: trong mỗi tầng, nối nút vừa lấy ra với nút **đứng đầu hàng đợi** (nếu vẫn còn trong tầng).

In [15]:
def connect_bfs(root: Optional[Node]) -> Optional[Node]:
    if not root:
        return root
    q = deque([root])
    while q:
        size = len(q)
        for i in range(size):
            node = q.popleft()
            node.next = q[0] if i < size - 1 else None
            if node.left:
                q.append(node.left)
                q.append(node.right)
    return root

print(serialize_next(connect_bfs(build_perfect([1, 2, 3, 4, 5, 6, 7]))))

[1, '#', 2, 3, '#', 4, 5, 6, 7, '#']


`q[0]` sau khi `popleft` chính là nút kế tiếp trong tầng (nếu `i < size − 1`). Bộ nhớ `O(w)` — tầng cuối có `2^(k−1)` nút, tức gần `n/2`.

# 5. Cách 2: Đi ngang trên tầng đã nối — `O(1)` bộ nhớ

Khi tầng `d` đã được nối bằng `next`, nó **chính là một danh sách liên kết**. Đi dọc danh sách đó và nối tầng `d + 1`:

Với mỗi nút `cur` ở tầng `d`:

- **Cùng cha:** `cur.left.next = cur.right`.
- **Khác cha:** nếu `cur.next` tồn tại: `cur.right.next = cur.next.left`.

```text
tầng d:     2 ------> 3
           / \       / \
tầng d+1: 4 → 5 ---→ 6 → 7
          (cùng cha) (khác cha: 5.next = 2.next.left = 6)
```

Bắt đầu từ `leftmost = root`; xong tầng thì `leftmost = leftmost.left`. Dừng khi tới tầng lá (`leftmost.left is None`).

In [16]:
def connect_o1(root: Optional[Node]) -> Optional[Node]:
    leftmost = root
    while leftmost and leftmost.left:
        cur = leftmost
        while cur:
            cur.left.next = cur.right
            if cur.next:
                cur.right.next = cur.next.left
            cur = cur.next
        leftmost = leftmost.left
    return root

print(serialize_next(connect_o1(build_perfect(list(range(1, 16))))))

[1, '#', 2, 3, '#', 4, 5, 6, 7, '#', 8, 9, 10, 11, 12, 13, 14, 15, '#']


# 6. Mô phỏng

In [17]:
def trace_o1(values):
    root = build_perfect(values)
    leftmost, depth = root, 0
    while leftmost and leftmost.left:
        cur = leftmost
        print(f"Đang ở tầng {depth} (đi theo next), nối tầng {depth + 1}:")
        while cur:
            cur.left.next = cur.right
            msg = f"   cur = {cur.val}: {cur.left.val}.next = {cur.right.val}"
            if cur.next:
                cur.right.next = cur.next.left
                msg += f", {cur.right.val}.next = {cur.next.val}.left = {cur.next.left.val}"
            print(msg)
            cur = cur.next
        leftmost = leftmost.left
        depth += 1
    print("Kết quả:", serialize_next(root))
    return root

_ = trace_o1([1, 2, 3, 4, 5, 6, 7])

Đang ở tầng 0 (đi theo next), nối tầng 1:
   cur = 1: 2.next = 3
Đang ở tầng 1 (đi theo next), nối tầng 2:
   cur = 2: 4.next = 5, 5.next = 3.left = 6
   cur = 3: 6.next = 7
Kết quả: [1, '#', 2, 3, '#', 4, 5, 6, 7, '#']


# 7. Cách 3: Đệ quy

`connect(node)` nối hai con của `node` (cùng cha), và con phải với con trái của `node.next` (khác cha), rồi đệ quy xuống. Vì đi **từ trên xuống**, `node.next` luôn đã được nối khi ta cần.

In [18]:
def connect_rec(root: Optional[Node]) -> Optional[Node]:
    def go(node):
        if not node or not node.left:
            return
        node.left.next = node.right
        if node.next:
            node.right.next = node.next.left
        go(node.left)
        go(node.right)
    go(root)
    return root

print(serialize_next(connect_rec(build_perfect([1, 2, 3, 4, 5, 6, 7]))))

[1, '#', 2, 3, '#', 4, 5, 6, 7, '#']


Độ sâu đệ quy bằng chiều cao cây `log2(n+1)` ≤ 12 — đề cho phép coi là không tính vào bộ nhớ phụ.

# 8. Phiên bản đúng format LeetCode

In [19]:
class Solution:
    def connect(self, root: 'Optional[Node]') -> 'Optional[Node]':
        leftmost = root
        while leftmost and leftmost.left:
            cur = leftmost
            while cur:
                cur.left.next = cur.right
                if cur.next:
                    cur.right.next = cur.next.left
                cur = cur.next
            leftmost = leftmost.left
        return root

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<10} PASS  {actual}")


def expected_next(values):
    out, i, width = [], 0, 1
    while i < len(values):
        out += values[i:i + width] + ["#"]
        i += width
        width *= 2
    return out


def next_targets(root):
    """Map val -> val của next (hoặc None), để so khớp chi tiết."""
    res, q = {}, deque([root] if root else [])
    while q:
        n = q.popleft()
        res[n.val] = n.next.val if n.next else None
        if n.left:
            q.append(n.left); q.append(n.right)
    return res


sol = Solution()
check("Ví dụ 1", serialize_next(sol.connect(build_perfect([1, 2, 3, 4, 5, 6, 7]))), [1, "#", 2, 3, "#", 4, 5, 6, 7, "#"])
check("Ví dụ 2", serialize_next(sol.connect(build_perfect([]))), [])
check("Một nút", serialize_next(sol.connect(build_perfect([1]))), [1, "#"])

for k in range(0, 13):
    vals = list(range(2 ** k - 1))
    a = next_targets(sol.connect(build_perfect(vals)))
    b = next_targets(connect_bfs(build_perfect(vals)))
    c = next_targets(connect_rec(build_perfect(vals)))
    assert a == b == c
    assert serialize_next(sol.connect(build_perfect(vals))) == expected_next(vals)
print("Mọi cây hoàn hảo tới 12 tầng (4095 nút): 3 cách cho cùng kết quả, đúng thứ tự tầng")

Ví dụ 1    PASS  [1, '#', 2, 3, '#', 4, 5, 6, 7, '#']
Ví dụ 2    PASS  []
Một nút    PASS  [1, '#']


Mọi cây hoàn hảo tới 12 tầng (4095 nút): 3 cách cho cùng kết quả, đúng thứ tự tầng


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** khi bắt đầu xử lý tầng `d` (bằng `leftmost`), mọi con trỏ `next` của tầng `d` đã đúng.

- Với cây hoàn hảo, tầng `d + 1` gồm, theo thứ tự trái sang phải: `c1.left, c1.right, c2.left, c2.right, …` với `c1, c2, …` là tầng `d` theo `next`.
- Hai loại cặp liền kề: `(ci.left, ci.right)` — cùng cha; `(ci.right, c(i+1).left)` — khác cha. Thuật toán nối đúng cả hai, và `c_last.right.next` giữ `None`.

```text
Time:  O(n)  — mỗi nút được cur đi qua một lần
Space: O(1)  (cách 2) | O(w) (BFS) | O(h) đệ quy (cách 3)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Chỉ nối anh em cùng cha

In [21]:
def wrong_siblings_only(root):
    def go(node):
        if not node or not node.left:
            return
        node.left.next = node.right                # thiếu nối khác cha
        go(node.left)
        go(node.right)
    go(root)
    return root

print("Sai :", serialize_next(wrong_siblings_only(build_perfect([1, 2, 3, 4, 5, 6, 7]))))

Sai : [1, '#', 2, 3, '#', 4, 5, '#']


Tầng cuối chỉ thấy `4, 5` — `5.next` vẫn là `None` vì `5` và `6` khác cha.

## Lỗi 2: Đệ quy từ dưới lên (nối tầng con trước khi tầng cha có `next`)

In [22]:
def wrong_bottom_up(root):
    def go(node):
        if not node or not node.left:
            return
        go(node.left)                              # SAI: đệ quy trước khi nối
        go(node.right)
        node.left.next = node.right
        if node.next:
            node.right.next = node.next.left
    go(root)
    return root

print("Sai :", serialize_next(wrong_bottom_up(build_perfect([1, 2, 3, 4, 5, 6, 7]))))

Sai : [1, '#', 2, 3, '#', 4, 5, '#']


Khi xử lý `2`, `2.next` chưa được gán (vì `1` chưa xử lý) → không nối được `5 → 6`.

## Lỗi 3: Dùng cách 2 cho cây **không hoàn hảo**

`cur.left.next = cur.right` giả định mọi nút trong có đủ hai con; với cây bất kỳ sẽ lỗi `AttributeError` hoặc nối sai — đó là bài #117.

## Lỗi 4: BFS nối cả nút cuối tầng với nút đầu tầng sau

Nếu viết `node.next = q[0] if q else None` (không kiểm tra `i < size − 1`), nút cuối tầng sẽ trỏ sang nút đầu tầng dưới.

In [23]:
def wrong_bfs_cross_level(root):
    q = deque([root])
    while q:
        for _ in range(len(q)):
            node = q.popleft()
            if node.left:
                q.append(node.left); q.append(node.right)
            node.next = q[0] if q else None        # SAI
    return root

r = wrong_bfs_cross_level(build_perfect([1, 2, 3, 4, 5, 6, 7]))
print("Sai : 1.next =", r.next.val if r.next else None, "| 3.next =", r.right.next.val if r.right.next else None)

Sai : 1.next = 2 | 3.next = 4


# 11. Mẹo nhận diện pattern

- **"Nối các nút cùng tầng"** → BFS theo tầng; follow-up `O(1)`: dùng **tầng đã nối làm danh sách liên kết** để duyệt ngang.
- Cây hoàn hảo → hai loại nối (cùng cha / khác cha). Cây bất kỳ (#117) → dùng **dummy** cho tầng dưới.

Bài liên quan: **#117 Populating Next Right Pointers II**, **#102 Level Order**, **#199 Right Side View**, **#2415 Reverse Odd Levels**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với cây hoàn hảo 4 tầng `1..15`, nút `7` có `next` là gì? Nút `11`? Nút `15`?

## Bài 2 — Điền chỗ trống (cách 2)

In [24]:
def exercise_2_connect(root: Optional[Node]) -> Optional[Node]:
    leftmost = root
    while leftmost and leftmost.left:
        cur = leftmost
        while cur:
            # TODO: nối cùng cha; nối khác cha nếu có cur.next; đi tiếp
            pass
        leftmost = leftmost.left
    return root


# assert serialize_next(exercise_2_connect(build_perfect([1, 2, 3, 4, 5, 6, 7]))) == [1, "#", 2, 3, "#", 4, 5, 6, 7, "#"]
# assert serialize_next(exercise_2_connect(build_perfect([]))) == []

## Bài 3 — Biến thể: dùng `next` để lấy level order

Sau khi `connect`, viết `level_order_via_next(root)` trả về list các tầng (như #102) **không dùng hàng đợi**.

In [25]:
def exercise_3_level_order_via_next(root: Optional[Node]) -> List[List[int]]:
    # TODO
    pass


# assert exercise_3_level_order_via_next(Solution().connect(build_perfect([1, 2, 3, 4, 5, 6, 7]))) == [[1], [2, 3], [4, 5, 6, 7]]
# assert exercise_3_level_order_via_next(None) == []

## Bài 4 — Biến thể: nối `next` theo vòng (tầng cuối nối về đầu tầng)

Như bài chính nhưng nút **cuối** mỗi tầng trỏ về nút **đầu** tầng đó (mỗi tầng thành danh sách vòng). Bộ nhớ `O(1)`.

In [26]:
def exercise_4_connect_circular(root: Optional[Node]) -> Optional[Node]:
    # TODO
    pass


# r = exercise_4_connect_circular(build_perfect([1, 2, 3, 4, 5, 6, 7])); assert r.next is r and r.right.next is r.left and r.right.right.next is r.left.left

## Bài 5 — Mini challenge: tính `next` mà không dựng cây

Cây hoàn hảo đánh số theo level-order từ 0 (`0 .. 2^k − 2`). Viết hàm `next_index(i, k)` trả chỉ số của `next` của nút `i`, hoặc `-1`, trong `O(1)`.

```python
next_index(1, 3) -> 2     # nút 1 (tầng 1) -> nút 2
next_index(2, 3) -> -1    # cuối tầng 1
next_index(4, 3) -> 5
```

Gợi ý nhanh: nút cuối tầng có chỉ số `2^(d+1) − 2`; tầng của `i` là `(i + 1).bit_length() − 1`.

In [27]:
def exercise_5_next_index(i: int, k: int) -> int:
    # TODO
    pass


# assert exercise_5_next_index(1, 3) == 2
# assert exercise_5_next_index(2, 3) == -1
# assert exercise_5_next_index(4, 3) == 5
# assert exercise_5_next_index(0, 1) == -1

# 13. Gợi ý

**Bài 1**
- Mức 1: tầng 2 là `4..7`; `7` cuối tầng.

**Bài 3**
- Mức 1: `leftmost = root`; với mỗi tầng đi theo `next` và gom giá trị; `leftmost = leftmost.left`.

**Bài 4**
- Mức 1: chạy bài chính; sau đó với mỗi tầng, đi tới nút cuối và đặt `last.next = leftmost`.
- Mức 2: cẩn thận — khi đã thành vòng, dừng đi ngang khi quay lại `leftmost`.

**Bài 5**
- Mức 1: `d = (i + 1).bit_length() - 1`; nếu `i == 2**(d+1) - 2` thì `-1`, ngược lại `i + 1`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [28]:
nt = next_targets(sol.connect(build_perfect(list(range(1, 16)))))
print("7 ->", nt[7], "| 11 ->", nt[11], "| 15 ->", nt[15])

7 -> None | 11 -> 12 | 15 -> None


## Đáp án Bài 2

In [29]:
def answer_2_connect(root: Optional[Node]) -> Optional[Node]:
    return connect_o1(root)

assert serialize_next(answer_2_connect(build_perfect([1, 2, 3, 4, 5, 6, 7]))) == [1, "#", 2, 3, "#", 4, 5, 6, 7, "#"]
assert serialize_next(answer_2_connect(build_perfect([]))) == []
print("Bài 2 PASS (giống connect_o1 ở mục 5)")

Bài 2 PASS (giống connect_o1 ở mục 5)


## Đáp án Bài 3

In [30]:
def answer_3_level_order_via_next(root: Optional[Node]) -> List[List[int]]:
    out, leftmost = [], root
    while leftmost:
        level, node = [], leftmost
        while node:
            level.append(node.val)
            node = node.next
        out.append(level)
        leftmost = leftmost.left
    return out

assert answer_3_level_order_via_next(Solution().connect(build_perfect([1, 2, 3, 4, 5, 6, 7]))) == [[1], [2, 3], [4, 5, 6, 7]]
assert answer_3_level_order_via_next(None) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [31]:
def answer_4_connect_circular(root: Optional[Node]) -> Optional[Node]:
    connect_o1(root)
    leftmost = root
    while leftmost:
        node = leftmost
        while node.next:
            node = node.next
        node.next = leftmost
        leftmost = leftmost.left
    return root

r = answer_4_connect_circular(build_perfect([1, 2, 3, 4, 5, 6, 7])); assert r.next is r and r.right.next is r.left and r.right.right.next is r.left.left
print("Bài 4 PASS")

Bài 4 PASS


Thứ tự quan trọng: phải nối **xong mọi tầng** bằng `connect_o1` trước, rồi mới biến từng tầng thành vòng — nếu biến tầng trên thành vòng trước, vòng `while cur` của `connect_o1` sẽ không bao giờ dừng.

## Đáp án Bài 5

In [32]:
def answer_5_next_index(i: int, k: int) -> int:
    d = (i + 1).bit_length() - 1
    return -1 if i == 2 ** (d + 1) - 2 else i + 1

assert answer_5_next_index(1, 3) == 2
assert answer_5_next_index(2, 3) == -1
assert answer_5_next_index(4, 3) == 5
assert answer_5_next_index(0, 1) == -1
for k in range(1, 9):
    vals = list(range(2 ** k - 1))
    nt = next_targets(sol.connect(build_perfect(vals)))
    assert all(answer_5_next_index(i, k) == (nt[i] if nt[i] is not None else -1) for i in vals)
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Cây hoàn hảo: tầng đã nối là một danh sách liên kết. Đi dọc nó, nối tầng dưới bằng hai quy tắc `cur.left.next = cur.right` và `cur.right.next = cur.next.left`. `O(n)` thời gian, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Nối cả cặp **cùng cha** và **khác cha**?
- [ ] Nút cuối mỗi tầng giữ `next = None`?
- [ ] Đi từ trên xuống (tầng trên đã nối trước)?
- [ ] `root = None` → trả `None`?